# Explorando outras arquiteturas

Este *notebook* tem como objetivo mostrar a implementação de PINNs em outras quiteturas que não MPL

**Autor:** Edélio Gabriel Magalhães de Jesus.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath("../scripts"))

import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

In [2]:
import torch.nn as nn
import torch.optim as optim
import torch
import plotly.graph_objects as go
from geral_functions import PINN, sample_collocation_lshape, sample_boundary_lshape
from ex01_pinn_direct_stacionary_vannila import analytical_solution, evaluate, train

In [3]:
# Definição do local onde o código serpa executado. Por padrão, gpu
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {DEVICE}')

Usando: cpu


In [4]:
# ------------------------------------------------------------
# Seção 1 — PINN MLP baseline — Poisson L-shape
# ------------------------------------------------------------

# modelo
modelo_mlp = PINN(n_inputs=2, n_outputs=1,
                  n_hidden=64, n_layers=4,
                  activation=nn.Tanh).to(DEVICE)

# pontos
X_col        = sample_collocation_lshape(2000, DEVICE)
X_bc, U_bc   = sample_boundary_lshape(100, DEVICE)

# loss — única coisa nova: o resíduo de Poisson com f=1
def loss_poisson(modelo, X_col, X_bc, U_bc, w_bc=10.0):
    u    = modelo(X_col)
    grads = torch.autograd.grad(u, X_col,
                                grad_outputs=torch.ones_like(u),
                                create_graph=True)[0]
    u_xx = torch.autograd.grad(grads[:, 0:1], X_col,
                               grad_outputs=torch.ones_like(grads[:, 0:1]),
                               create_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(grads[:, 1:2], X_col,
                               grad_outputs=torch.ones_like(grads[:, 1:2]),
                               create_graph=True)[0][:, 1:2]

    loss_pde = ((-u_xx - u_yy) - 1).pow(2).mean()
    loss_bc  = (modelo(X_bc) - U_bc).pow(2).mean()
    return loss_pde + w_bc * loss_bc, loss_pde, loss_bc

In [5]:
N_EPOCHS = 2000
LR = 1e-4
OPTIMIZER = torch.optim.Adam(modelo_mlp.parameters(), lr=LR)

In [6]:
history = {'loss': [], 'loss_data': [], 'loss_pde': []}

for epoch in range(N_EPOCHS):

    OPTIMIZER.zero_grad()

    loss, loss_data, loss_pde = loss_poisson(
        modelo_mlp, X_col, X_bc, U_bc
    )

    loss.backward()
    OPTIMIZER.step()

    history['loss'].append(loss.item())
    history['loss_data'].append(loss_data.item())
    history['loss_pde'].append(loss_pde.item())

    if epoch % 100 == 0:
        print(f'Epoch {epoch:05d} | Loss: {loss.item():.2e} | '
                f'Loss data: {loss_data.item():.2e} | '
                f'Loss PDE: {loss_pde.item():.2e}')

Epoch 00000 | Loss: 9.93e-01 | Loss data: 9.80e-01 | Loss PDE: 1.31e-03


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Seção 2 — PhyGeoNet — Poisson em domínio L-shape
# ============================================================

# ------------------------------------------------------------
# 2.1 Grade e máscara do L-shape
# ------------------------------------------------------------

H, W = 64, 64  # resolução da grade

# coordenadas
x1d = torch.linspace(0, 1, W, device=DEVICE)
y1d = torch.linspace(0, 1, H, device=DEVICE)
YY, XX = torch.meshgrid(y1d, x1d, indexing="ij")  # (H, W)

# máscara: True = dentro do domínio L-shape
mask = ~((XX > 0.5) & (YY > 0.5))   # (H, W)

# entrada da CNN: canais x e y empilhados → (1, 2, H, W)
X_grid = torch.stack([XX, YY], dim=0).unsqueeze(0)  # (1, 2, H, W)
X_grid = X_grid.to(DEVICE)

In [ ]:
# ------------------------------------------------------------
# 2.2 Arquitetura PhyGeoNet
#
# Encoder-decoder CNN simples:
#   - Encoder: extrai features espaciais com convoluções + pooling
#   - Decoder: reconstrói o campo u na resolução original
#
# Entrada:  (B, 2, H, W)  — canais: coordenadas (x, y)
# Saída:    (B, 1, H, W)  — campo u(x, y)
# ------------------------------------------------------------

class PhyGeoNet(nn.Module):
    def __init__(self):
        super().__init__()

        # encoder
        self.enc1 = self._bloco(2,  32)   # (B,  2, H,   W)   → (B, 32, H,   W)
        self.enc2 = self._bloco(32, 64)   # (B, 32, H/2, W/2) → (B, 64, H/2, W/2)
        self.enc3 = self._bloco(64, 128)  # (B, 64, H/4, W/4) → (B,128, H/4, W/4)

        self.pool = nn.MaxPool2d(2)

        # decoder com upsampling bilinear
        self.dec3 = self._bloco(128, 64)
        self.dec2 = self._bloco(64,  32)
        self.dec1 = self._bloco(32,  16)

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

        # cabeça de saída
        self.out = nn.Conv2d(16, 1, kernel_size=1)

    def _bloco(self, c_in, c_out):
        return nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1),
            nn.Tanh(),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1),
            nn.Tanh(),
        )

    def forward(self, x):
        # encoder
        e1 = self.enc1(x)             # (B, 32, H,   W)
        e2 = self.enc2(self.pool(e1)) # (B, 64, H/2, W/2)
        e3 = self.enc3(self.pool(e2)) # (B,128, H/4, W/4)

        # decoder
        d3 = self.dec3(self.up(e3))   # (B, 64, H/2, W/2)
        d2 = self.dec2(self.up(d3))   # (B, 32, H,   W)
        d1 = self.dec1(d2)            # (B, 16, H,   W)

        return self.out(d1)           # (B,  1, H,   W)


modelo_geo = PhyGeoNet().to(device)
print(f"Parâmetros treináveis: {sum(p.numel() for p in modelo_geo.parameters()):,}")

In [ ]:
# ------------------------------------------------------------
# 2.3 Derivadas por diferenças finitas via convolução
#
# Os filtros são fixos (não treináveis) e calculam ∂u/∂x e
# ∂²u/∂x² diretamente no campo em grade.
# ------------------------------------------------------------

h = 1.0 / (W - 1)  # espaçamento da grade

# filtros 1D → kernels 2D para Conv2d
# ∂/∂x — diferença centrada horizontal
k_dx = torch.tensor([[[[0., 0., 0.],
                        [-1., 0., 1.],
                        [0., 0., 0.]]]], device=DEVICE) / (2 * h)

# ∂/∂y — diferença centrada vertical
k_dy = torch.tensor([[[[0., -1., 0.],
                        [0.,  0., 0.],
                        [0.,  1., 0.]]]], device=DEVICE) / (2 * h)

# ∂²/∂x² — diferença centrada segunda ordem
k_dxx = torch.tensor([[[[0., 0., 0.],
                         [1., -2., 1.],
                         [0., 0., 0.]]]], device=DEVICE) / (h ** 2)

# ∂²/∂y² — diferença centrada segunda ordem
k_dyy = torch.tensor([[[[0.,  1., 0.],
                         [0., -2., 0.],
                         [0.,  1., 0.]]]], device=DEVICE) / (h ** 2)


def laplaciano(u):
    """Calcula ∇²u = ∂²u/∂x² + ∂²u/∂y² via convolução com padding de borda."""
    u_xx = nn.functional.conv2d(u, k_dxx, padding=1)
    u_yy = nn.functional.conv2d(u, k_dyy, padding=1)
    return u_xx + u_yy

In [ ]:
# ------------------------------------------------------------
# 2.4 Função de perda PhyGeoNet
#
# Loss_física:   resíduo de -∇²u = 1 nos pontos do L-shape
# Loss_contorno: u = 0 na fronteira do L-shape
#
# A máscara separa interior de fronteira diretamente no grid.
# ------------------------------------------------------------

# máscara de fronteira: borda do L-shape no grid
def mascara_fronteira(mask, H, W):
    """
    Identifica células do grid que estão na fronteira do L-shape:
    dentro do domínio mas com pelo menos um vizinho fora.
    """
    mask_f  = torch.zeros(H, W, dtype=torch.bool, device=DEVICE)
    m       = mask.float()

    # célula é fronteira se está no domínio e tem vizinho fora
    interior = (
        m[1:-1, 1:-1] *
        (1 - m[:-2, 1:-1]) +   # vizinho de cima fora
        m[1:-1, 1:-1] *
        (1 - m[2:,  1:-1]) +   # vizinho de baixo fora
        m[1:-1, 1:-1] *
        (1 - m[1:-1, :-2]) +   # vizinho da esquerda fora
        m[1:-1, 1:-1] *
        (1 - m[1:-1, 2:])      # vizinho da direita fora
    )
    mask_f[1:-1, 1:-1] = interior > 0

    # bordas externas do grid também são fronteira se estão no domínio
    mask_f[0,  :]  = mask[0,  :]
    mask_f[-1, :]  = mask[-1, :]
    mask_f[:,  0]  = mask[:,  0]
    mask_f[:, -1]  = mask[:, -1]

    return mask_f


mask_bc  = mascara_fronteira(mask, H, W)   # fronteira
mask_int = mask & ~mask_bc                 # interior puro


def loss_phygeonet(modelo, X_grid, mask_int, mask_bc, lambda_bc=10.0):
    u = modelo(X_grid)                     # (1, 1, H, W)
    u_sq = u.squeeze()                     # (H, W)

    lap = laplaciano(u).squeeze()          # (H, W)

    # resíduo de -∇²u = 1 no interior
    residuo  = (-lap - 1.0)[mask_int]
    loss_pde = (residuo ** 2).mean()

    # u = 0 na fronteira
    loss_bc  = (u_sq[mask_bc] ** 2).mean()

    return loss_pde + lambda_bc * loss_bc, loss_pde, loss_bc

In [ ]:
# ------------------------------------------------------------
# 2.5 Treinamento
# ------------------------------------------------------------

otimizador_geo = torch.optim.Adam(modelo_geo.parameters(), lr=1e-3)
scheduler_geo  = torch.optim.lr_scheduler.StepLR(
                     otimizador_geo, step_size=500, gamma=0.5)
n_epocas_geo   = 3000
historico_geo  = {"total": [], "pde": [], "bc": []}

for epoca in range(n_epocas_geo):
    otimizador_geo.zero_grad()
    loss, l_pde, l_bc = loss_phygeonet(
        modelo_geo, X_grid, mask_int, mask_bc)
    loss.backward()
    otimizador_geo.step()
    scheduler_geo.step()

    historico_geo["total"].append(loss.item())
    historico_geo["pde"].append(l_pde.item())
    historico_geo["bc"].append(l_bc.item())

    if (epoca + 1) % 300 == 0:
        print(f"Época {epoca+1:5d} | Loss: {loss.item():.2e} "
              f"| PDE: {l_pde.item():.2e} | BC: {l_bc.item():.2e}")

In [ ]:
# ------------------------------------------------------------
# 2.6 Resultados e comparação com MLP
# ------------------------------------------------------------

modelo_geo.eval()
with torch.no_grad():
    U_geo = modelo_geo(X_grid).squeeze().cpu().numpy()  # (H, W)

# aplica máscara — fora do domínio vira NaN
U_geo_plot = np.where(mask.cpu().numpy(), U_geo, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# PhyGeoNet
im0 = axes[0].imshow(U_geo_plot, origin="lower",
                     extent=[0, 1, 0, 1], cmap="viridis")
plt.colorbar(im0, ax=axes[0], label="u(x,y)")
axes[0].set_title("PhyGeoNet — Solução prevista")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")

# curvas de loss comparadas
axes[1].semilogy(historico_mlp["total"], label="MLP — total",  color="steelblue")
axes[1].semilogy(historico_geo["total"], label="CNN — total",  color="darkorange")
axes[1].semilogy(historico_mlp["pde"],   label="MLP — PDE",   color="steelblue",   ls="--")
axes[1].semilogy(historico_geo["pde"],   label="CNN — PDE",   color="darkorange",  ls="--")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Loss")
axes[1].set_title("Comparação de treinamento")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# --- métricas ---
res_geo = (-laplaciano(modelo_geo(X_grid)).squeeze().detach()
            - 1.0)[mask_int].abs().cpu().numpy()

print(f"\n{'':30s} {'MLP':>12s} {'PhyGeoNet':>12s}")
print(f"{'Resíduo médio |−∇²u − 1|':30s} "
      f"{residuo_mlp.mean():>12.4e} {res_geo.mean():>12.4e}")
print(f"{'Resíduo máx  |−∇²u − 1|':30s} "
      f"{residuo_mlp.max():>12.4e} {res_geo.max():>12.4e}")
print(f"{'Parâmetros':30s} "
      f"{sum(p.numel() for p in modelo_mlp.parameters()):>12,} "
      f"{sum(p.numel() for p in modelo_geo.parameters()):>12,}")